In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, fbeta_score,
    confusion_matrix, classification_report
)

# =========================
# CROSS-SELL (komplett)
# =========================
# Erklärung: Ziel = "Cross-Kandidat" (Top-20% Käufer in EINER Produktkategorie).
# Danach:
# - Leakage raus (die Ziel-Kategorie-Ausgaben dürfen NICHT als Feature rein)
# - Modell trainieren (LogReg + class_weight)
# - Threshold optimieren (Precision-first)
# - Cross-Score + Flag für alle Kunden erstellen

# 1) Daten laden
df = pd.read_csv("Marktkampagne.csv")

# 2) Mini-Clean
df["Datum_Kunde"] = pd.to_datetime(df["Datum_Kunde"], dayfirst=True, errors="raise")
df["Einkommen_fehlte"] = df["Einkommen"].isna().astype(int)
df["Einkommen"] = df["Einkommen"].fillna(df["Einkommen"].median())

# 3) Kundendauer als Feature
stichtag = df["Datum_Kunde"].max()
df["Kundendauer_Tage"] = (stichtag - df["Datum_Kunde"]).dt.days

# 4) Ziel-Kategorie auswählen (automatisch)
# Erklärung: Wir suchen eine passende "Ausgaben_*" oder "Mnt*" Spalte.
prioritaet = [
    "Ausgaben_Fisch", "Ausgaben_Wein", "Ausgaben_Fleisch", "Ausgaben_Gold", "Ausgaben_Süßes",
    "MntFishProducts", "MntWines", "MntMeatProducts", "MntGoldProds", "MntSweetProducts"
]

spend_cols = [c for c in df.columns if c.lower().startswith("ausgaben_")]
if not spend_cols:
    spend_cols = [c for c in df.columns if c.startswith("Mnt")]

cross_col = next((c for c in prioritaet if c in df.columns), None)
if cross_col is None:
    cross_col = spend_cols[0]  # fallback

print("Cross-Zielspalte:", cross_col)

# 5) Cross-Target bauen (Top-20% in dieser Kategorie)
# Erklärung: Wenn es genug Nicht-Nullen gibt -> quantile(0.80) über alle.
# Sonst -> Top-20% innerhalb der Nicht-Nullen (damit das Label nicht nur "0 vs 0" wird).
nonzero = df[cross_col] > 0

if nonzero.mean() >= 0.20:
    cross_thr = df[cross_col].quantile(0.80)
    df["Ziel_Cross"] = (df[cross_col] >= cross_thr).astype(int)
else:
    cross_thr = df.loc[nonzero, cross_col].quantile(0.80)
    df["Ziel_Cross"] = ((df[cross_col] >= cross_thr) & nonzero).astype(int)

print("Cross-Schwelle:", float(cross_thr))
print("Cross-Rate (Anteil 1):", df["Ziel_Cross"].mean())

# 6) Features bauen (Leakage vermeiden!)
# Erklärung: cross_col muss raus, weil es das Ziel direkt verrät.
drop_cols = ["ID", "Datum_Kunde", "Ziel_Cross", cross_col]
X_cross = df.drop(columns=[c for c in drop_cols if c in df.columns])
y_cross = df["Ziel_Cross"]

cat_cols = X_cross.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X_cross.columns if c not in cat_cols]

# 7) Train/Test Split
Xtr, Xte, ytr, yte = train_test_split(
    X_cross, y_cross, test_size=0.2, random_state=42, stratify=y_cross
)

# 8) Modell (Baseline)
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

model_cross = Pipeline(steps=[
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced"))
])

model_cross.fit(Xtr, ytr)
proba_cross = model_cross.predict_proba(Xte)[:, 1]

print("\nROC-AUC:", roc_auc_score(yte, proba_cross))
print("PR-AUC:", average_precision_score(yte, proba_cross))

# 9) Threshold-Optimierung (Precision-first)
# Erklärung: Bei Cross-Sell ist oft Precision wichtig (nicht zu viele falsche Angebote).
thresholds = np.linspace(0.05, 0.95, 19)

best = {"t": None, "precision": None, "recall": None, "f1": None, "f0_5": -1, "positives_pred": None}
for t in thresholds:
    pred = (proba_cross >= t).astype(int)
    p = precision_score(yte, pred, zero_division=0)
    r = recall_score(yte, pred, zero_division=0)
    f1 = f1_score(yte, pred, zero_division=0)
    f05 = fbeta_score(yte, pred, beta=0.5, zero_division=0)

    if f05 > best["f0_5"]:
        best = {"t": float(t), "precision": p, "recall": r, "f1": f1, "f0_5": f05, "positives_pred": int(pred.sum())}

pred_best = (proba_cross >= best["t"]).astype(int)

print("\nBeste Schwelle (F0.5):", best["t"])
print({"precision": best["precision"], "recall": best["recall"], "f1": best["f1"], "f0_5": best["f0_5"], "positives_pred": best["positives_pred"]})
print("Confusion Matrix:\n", confusion_matrix(yte, pred_best))
print("\nReport:\n", classification_report(yte, pred_best))

# 10) Cross-Score + Flag für alle Kunden
proba_all = model_cross.predict_proba(X_cross)[:, 1]
df["Cross_Score"] = proba_all
df["Cross_Flag"] = (df["Cross_Score"] >= best["t"]).astype(int)

# 11) Top-Liste anzeigen
cross_top = df.loc[df["Cross_Flag"] == 1, ["ID", "Cross_Score"]].sort_values("Cross_Score", ascending=False)

print("\nTop Cross-Kandidaten (erste 20):")
display(cross_top.head(20))


Cross-Zielspalte: Ausgaben_Fisch
Cross-Schwelle: 65.0
Cross-Rate (Anteil 1): 0.20625

ROC-AUC: 0.9356680508060576
PR-AUC: 0.7067689500685592

Beste Schwelle (F0.5): 0.75
{'precision': 0.7127659574468085, 'recall': 0.7282608695652174, 'f1': 0.7204301075268817, 'f0_5': 0.7158119658119658, 'positives_pred': 94}
Confusion Matrix:
 [[329  27]
 [ 25  67]]

Report:
               precision    recall  f1-score   support

           0       0.93      0.92      0.93       356
           1       0.71      0.73      0.72        92

    accuracy                           0.88       448
   macro avg       0.82      0.83      0.82       448
weighted avg       0.88      0.88      0.88       448


Top Cross-Kandidaten (erste 20):


,ID,Cross_Score
27,5255,0.998831
1075,6421,0.998290
1601,5453,0.997310
252,10089,0.995640
723,10936,0.995222
360,7274,0.994787
591,7627,0.994668
1877,1399,0.994607
1921,3283,0.994090
456,4947,0.993707


In [ ]:
# =========================
# Erklärung (Cross-Sell Ergebnis – einfach)
# =========================
# 1) Ziel/Label (was ist "Cross" hier?):
#    - Cross-Zielspalte: Ausgaben_Fisch
#    - Wir nennen einen Kunden "Cross-Kandidat", wenn er zu den Top-20% bei Ausgaben_Fisch gehört.
#    - Cross-Schwelle: 65.0 bedeutet: ab 65 (Einheiten der Spalte) zählt der Kunde als Top-20%-Käufer.
#    - Cross-Rate (Anteil 1): 0.20625 -> ca. 20.6% der Kunden sind Cross-Kandidaten (nach dieser Definition).
#
# 2) Modellgüte (wie gut trennt das Modell generell?):
#    - ROC-AUC: 0.936 -> sehr gute Trennschärfe (das Modell kann Kandidaten gut von Nicht-Kandidaten unterscheiden).
#    - PR-AUC: 0.707 -> gut, speziell für die positive Klasse (Cross-Kandidat).
#
# 3) Optimierte Entscheidungsschwelle (wie streng markieren wir Kandidaten?):
#    - Beste Schwelle (F0.5): 0.75
#      -> Wir markieren nur Kunden als Cross-Kandidaten, wenn das Modell mindestens 75% Wahrscheinlichkeit sieht.
#      -> F0.5 bedeutet: Precision ist wichtiger als Recall (Cross-Angebote sollen eher "passen" als streuen).
#
# 4) Precision / Recall (was heißt das in normaler Sprache?):
#    - Precision (0.713): Von allen Kunden, die wir als Cross-Kandidaten markieren, sind ca. 71% wirklich Kandidaten.
#    - Recall (0.728): Wir finden ca. 73% aller echten Kandidaten (wir verpassen nur ca. 27%).
#    - positives_pred: 94 -> so viele Kunden würden wir im Test als Cross-Kandidaten markieren.
#
# 5) Confusion Matrix (konkret in Zahlen):
#    - [[329, 27],
#       [ 25, 67]]
#    - 67 = echte Kandidaten korrekt erkannt (TP)
#    - 27 = fälschlich als Kandidat markiert (FP)
#    - 25 = echte Kandidaten verpasst (FN)
#    - 329 = korrekt als Nicht-Kandidat erkannt (TN)
#
# 6) Kurzfazit:
#    - Das Modell ist insgesamt stark (ROC-AUC hoch).
#    - Mit Schwelle 0.75 bekommst du eine gute Balance:
#      - relativ wenig Fehlalarme (Precision ~71%)
#      - gleichzeitig gute Abdeckung (Recall ~73%)
#    - Praktisch heißt das: Du kannst Cross-Sell-Angebote für Fisch relativ gezielt ausspielen,
#      ohne zu viele unpassende Kunden anzuschreiben.
